In [2]:
import sys
import tqdm
import json 
import copy
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch
import numpy as np
from matplotlib import colors
import torchvision
import torch.nn.functional as F
import pandas as pd
from matplotlib.colors import ListedColormap
from typing import Dict, Tuple
from torchmetrics.functional import jaccard_index, precision, recall, f1_score 
from torchmetrics.functional.classification import binary_precision_recall_curve
from sklearn.metrics import average_precision_score, precision_recall_curve, PrecisionRecallDisplay, roc_auc_score
from torchmetrics.classification import PrecisionRecallCurve
from torchmetrics import MeanAbsoluteError, MeanSquaredError, R2Score, PearsonCorrCoef, MetricCollection

from src.utils import config_helper, get_wandb_runs
from src.data.get_dataloaders import get_dataloaders
from src.data.dataset import DeforestationForecastingDataset
from src.algorithm.models.get_model import  get_model
from src.algorithm.loss import get_loss_fn
from src.evaluate import evaluate

In [6]:
@torch.no_grad()
def predict_deforestation_within_k_years(
    model: torch.nn.Module,
    init_def_seq: torch.Tensor,
    elevation: torch.Tensor,
    K: int,
    device: torch.device,
    binarize_feedback: bool = True,
    fixed_window: bool = True
) -> torch.Tensor:
    """
    Autoregressively predict deforestation over K years, and return the per-pixel
    MAXIMUM probability predicted in any of the K years.
    """
    def_seq   = init_def_seq.clone().to(device).to(torch.float32)  # (B, T, 1, H, W)
    elevation = elevation.clone().to(device).to(torch.float32)     # (B, 1, H, W)
    B, T, _, H, W = def_seq.shape

    base_year_map = def_seq[:, -1, 0]                               # (B, H, W)
    ineligible_mask = (base_year_map == -1) | (base_year_map == 1)  # bool

    # Initialize with zeros so max works correctly
    p_out = torch.zeros((B, H, W), device=device, dtype=torch.float32)

    for _ in range(K):
        logits = model({'deforestation_map': def_seq, 'elevation': elevation})  # (B, T_current, 1, H, W)
        step_probs = torch.sigmoid(logits[:, -1, 0])                            # (B, H, W)

        # Update per-pixel maximum probability over time
        p_out = torch.maximum(p_out, step_probs)

        forest_mask = ~ineligible_mask
        new_event_mask = (step_probs >= 0.5) & forest_mask

        ineligible_mask |= new_event_mask

        if binarize_feedback:
            next_frame = torch.zeros_like(base_year_map, dtype=torch.float32)
            next_frame[ineligible_mask] = -1.0
            next_frame[new_event_mask]  = 1.0
        else:
            next_frame = step_probs.clone()
            next_frame[ineligible_mask] = -1.0

        next_frame_t = next_frame.unsqueeze(1).unsqueeze(1)  # (B, 1, 1, H, W)
        if fixed_window:
            def_seq = torch.cat([def_seq[:, 1:], next_frame_t], dim=1)
        else:
            def_seq = torch.cat([def_seq, next_frame_t], dim=1)

    return p_out


In [7]:
def split_inputs_for_multi_year(
    input_batch: Dict[str, torch.Tensor],
    T: int,
    K: int,
    device: torch.device
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Prepare inputs and ground truth for a K-year evaluation, using a batch that already
    contains T+K time steps.

    Inputs
    ------
    input_batch : dict
        Must contain:
          - 'deforestation_map': Tensor (B, T+K, 1, H, W)
              Per-year encoding:
                1  -> deforested THIS year (first time)
                0  -> forest
               -1  -> non-forest (after the first deforestation year, all later years remain -1)
          - 'elevation': Tensor (B, 1, H, W) (static; the model repeats it over time internally)
    T : int
        Number of context years fed to the model.
    K : int
        Number of future years to forecast and evaluate.
    device : torch.device
        Device on which tensors will reside.

    Returns
    -------
    init_def_seq : Tensor (B, T, 1, H, W)
        The first T frames; used as the model’s input sequence.
    future_truth_seq : Tensor (K, B, H, W), dtype=bool
        Per-year ground truth for the next K years: True only in the first deforestation year.
        (Kept for convenience; evaluation below uses the full future frames to build masks precisely.)
    elevation : Tensor (B, 1, H, W)
        Static elevation map (unmodified).
    base_year_map : Tensor (B, H, W)
        The map at the base year t0 (last of the first T frames). Used to build evaluation masks.
    """
    def_seq_all = input_batch['deforestation_map'].to(device).to(torch.float32)  # (B, T+K, 1, H, W)
    elevation   = input_batch['elevation'].to(device).to(torch.float32)         # (B, 1, H, W)
    B, X, _, H, W = def_seq_all.shape
    assert X >= T + K, f"Expected at least T+K frames, got X={X} (T={T}, K={K})."

    # First T frames are the model input; next K frames are the future ground truth.
    init_def_seq = def_seq_all[:, :T]                         # (B, T, 1, H, W)
    future_frames = def_seq_all[:, T:T+K, 0]                  # (B, K, H, W)
    future_truth_seq = (future_frames == 1).permute(1, 0, 2, 3)  # (K, B, H, W) bool

    # Base year (t0) is the last frame among the first T.
    base_year_map = init_def_seq[:, -1, 0]                    # (B, H, W)

    return init_def_seq, future_truth_seq, elevation, base_year_map




def compute_metrics(targets, preds, dataset_name, regression_metrics=True):

    preds = preds.to(dtype=torch.float16)
    targets = targets.to(dtype=torch.int16)

    mask = targets != -1
    valid_targets = targets[mask].flatten()
    valid_preds = preds[mask].flatten()
    
    
    if regression_metrics:
        binarized_preds = (preds >= 0.5).int() 
        pred_pos = ((binarized_preds == 1) & mask).sum(dim=(1,2)).float()
        targ_pos = ((targets == 1) & mask).sum(dim=(1,2)).float()
        # stack for torchmetrics
        # exclude tiles with no valid pixels
        has_valid_data = mask.sum(dim=(1,2)) > 0
        pred_pos, targ_pos = pred_pos[has_valid_data], targ_pos[has_valid_data]
        pred_pos = pred_pos.unsqueeze(1)
        targ_pos = targ_pos.unsqueeze(1)
        reg_metrics_coll = MetricCollection({
            f'{dataset_name}_MAE': MeanAbsoluteError(),
            f'{dataset_name}_MSE': MeanSquaredError(),
            f'{dataset_name}_RMSE': MeanSquaredError(squared=False),
            f'{dataset_name}_R2': R2Score(),
            f'{dataset_name}_PearsonR': PearsonCorrCoef(),
        })

        reg_metrics = reg_metrics_coll(pred_pos, targ_pos)


    metrics = {
        f'{dataset_name}_iou' : jaccard_index(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_f1' : f1_score(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_precision' : precision(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_recall' : recall(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_average_precision' : average_precision_score(valid_targets.flatten(), valid_preds.flatten()),
        f'{dataset_name}_ROC_AUC' : roc_auc_score(valid_targets.flatten(), valid_preds.flatten()),
    }

    if regression_metrics:
        metrics.update(reg_metrics)


    return metrics


@torch.no_grad()
def evaluate_deforestation_within_k_years(
    model: torch.nn.Module,
    dataloader,
    device: torch.device,
    T: int,
    K: int,
    binarize_feedback: bool = True,
    fixed_window: bool = True,
    dataset_name: str = 'val',
    log_to_wandb: bool = False,
) -> Dict[str, torch.Tensor]:
    """
    Evaluate whether each pixel is correctly predicted to be deforested at least once
    within the next K years, using your existing dataloader (no changes to its API).

    Evaluation mask policy (handles -1 correctly across years)
    ----------------------------------------------------------
    A pixel is evaluated if:
      - it is forest in the base year t0 (value 0), AND
      - within the K-year period, it is never marked as -1 BEFORE its first deforestation year.
        (If no deforestation occurs within K, then any -1 in those K years causes the pixel to be ignored.)

    Steps per batch
    ---------------
      1) Split: first T frames -> model input; next K frames -> per-year ground truth (and statuses).
      2) Run K-step autoregressive prediction -> cumulative binary prediction (within K years).
      3) Build cumulative ground truth (deforested in any of the next K years) and the **validity mask** described above.
      4) Apply the validity mask: pixels outside are set to -1 (ignored).
      5) Compute IoU, F1, precision, recall, and average precision using your original metric function.
    """
    model.eval()
    preds_batches, targets_batches = [], []

    for input_batch, _ in dataloader:
        input_batch = {k: v.to(device).to(torch.float32) for k, v in input_batch.items()}

        # 1) Prepare the initial T frames, future per-year truth (K frames), static elevation, and base-year map.
        init_def_seq, future_truth_seq, elevation, base_year_map = split_inputs_for_multi_year(
            input_batch=input_batch, T=T, K=K, device=device
        )

        # 2) Autoregressive prediction: cumulative binary prediction (1 if deforested in any of the next K years).
        pred_within_period = predict_deforestation_within_k_years(
            model=model,
            init_def_seq=init_def_seq,
            elevation=elevation,
            K=K,
            device=device,
            binarize_feedback=binarize_feedback,
            fixed_window=fixed_window,
        )  # (B,H,W) in {0,1}

        # 3) Build cumulative ground truth and precise validity mask using the full future frames.
        #    We need the raw statuses (1/0/-1) for each future year, not just the boolean event maps.
        future_frames = input_batch['deforestation_map'][:, T:T+K, 0]  # (B,K,H,W) in {1,0,-1}

        event_seq     = (future_frames == 1)     # (B,K,H,W) True only in the first deforestation year
        nonforest_seq = (future_frames == -1)    # (B,K,H,W)

        # Has an event happened by (and including) each year? (cumulative over time)
        cum_event = torch.cumsum(event_seq.to(torch.int8), dim=1).clamp(max=1).bool()  # (B,K,H,W)

        # Years strictly before the first event (or all years if no event occurs within K)
        pre_event = torch.cat([torch.zeros_like(cum_event[:, :1]), cum_event[:, :-1]], dim=1) == 0  # (B,K,H,W)

        # Invalid if any -1 appears before the first event (or in any year if no event occurs within K)
        invalid_before_event = (nonforest_seq & pre_event).any(dim=1)  # (B,H,W)

        # Base-year eligibility: evaluate only where t0 is forest
        base_forest_mask = (base_year_map == 0)  # (B,H,W)

        # Final validity mask
        valid_mask = base_forest_mask & (~invalid_before_event)  # (B,H,W)

        # Cumulative ground truth: deforested in any of the next K years
        target_within_period = event_seq.any(dim=1).to(torch.int16)  # (B,H,W) {0,1}

        # Apply ignore (-1) outside the valid region
        targets_masked = torch.where(
            valid_mask,
            target_within_period,
            torch.full_like(target_within_period, -1, dtype=torch.int16)
        )

        preds_batches.append(pred_within_period.detach().cpu())
        targets_batches.append(targets_masked.detach().cpu())

    preds   = torch.cat(preds_batches, dim=0)
    targets = torch.cat(targets_batches, dim=0)

    metrics = compute_metrics(targets=targets, preds=preds, dataset_name=f'{dataset_name}_by_{K}y')


    return metrics, preds, targets


## trying to get the identical metrics

In [8]:
import torch

def _to_static_elevation(elev: torch.Tensor) -> torch.Tensor:
    """
    Ensure elevation is (B,1,H,W).
    Accepts (B,1,H,W) or (B,T,1,H,W) and reduces the latter to (B,1,H,W).
    """
    if elev.ndim == 4:   # (B,1,H,W)
        return elev
    if elev.ndim == 5:   # (B,T,1,H,W)
        return elev[:, -1, 0]  # last time step, drop the singleton channel
    raise ValueError(f"Unexpected elevation shape: {tuple(elev.shape)}")

@torch.no_grad()
def debug_compare_k1_vs_single(
    model,
    dataloader_single,   # built with input_seq_len = T, covers years [start, start+T]
    dataloader_multi,    # built with input_seq_len = T+1, covers years [start, start+T+1]
    device: torch.device,
    T: int
):
    """
    Quick checks to understand why K=1 metrics may differ from original single-year metrics.

    What it does for ONE batch from each loader:
      1) Verifies the model sees IDENTICAL T input frames in both setups.
      2) Verifies the model produces IDENTICAL probabilities given identical inputs.
      3) Compares the masks/targets actually used by each evaluation path.
      4) Computes metrics in three ways to pinpoint the divergence:
         - 'orig'                 : original single-year target + original probs path
         - 'k1_same_target_prob'  : original single-year target + multi path probs (should match 'orig' if inputs equal)
         - 'k1_pipeline_prob'     : K=1 constructed target + base-year mask + multi path probs ( K=1 pipeline)
    """
    model.eval()

    # --- Fetch one batch from each dataloader ---
    (inp_single, labels_single) = next(iter(dataloader_single))
    (inp_multi,  _labels_multi) = next(iter(dataloader_multi))  # labels unused in K=1 pipeline

    # Move to device and cast
    defor_single = inp_single['deforestation_map'].to(device).float()  # (B,T,1,H,W)
    elev_single  = _to_static_elevation(inp_single['elevation'].to(device).float())  # (B,1,H,W)
    labels_single = labels_single.to(device).float()  # (B,1,1,H,W) per your original evaluate()

    defor_multi_all = inp_multi['deforestation_map'].to(device).float()  # (B,T+1,1,H,W)
    elev_multi      = _to_static_elevation(inp_multi['elevation'].to(device).float())  # (B,1,H,W)

    # Sanity: shapes
    B1, T1, _, H1, W1 = defor_single.shape
    B2, X2, _, H2, W2 = defor_multi_all.shape
    assert T1 == T, f"Single-year T mismatch: got {T1}, expected {T}"
    assert X2 >= T + 1, f"Multi-year expects at least T+1 frames, got {X2}"
    assert (H1, W1) == (H2, W2), "Spatial dims differ between loaders"
    B = min(B1, B2)
    # Align batch sizes if they differ (take the first B samples)
    defor_single = defor_single[:B]
    elev_single  = elev_single[:B]
    labels_single = labels_single[:B]
    defor_multi_all = defor_multi_all[:B]
    elev_multi      = elev_multi[:B]

    # --- 1) Check that the model sees the SAME T input frames in both setups ---
    # Multi-year path uses only the FIRST T frames as context (K=1).
    defor_multi_T = defor_multi_all[:, :T]  # (B,T,1,H,W)

    same_inputs = torch.allclose(defor_single, defor_multi_T, atol=0, rtol=0)
    print(f"[Check] Input deforestation frames identical (single vs multi first T): {bool(same_inputs)}")
    same_elev   = torch.allclose(elev_single, elev_multi, atol=0, rtol=0)
    print(f"[Check] Elevation identical (single vs multi): {bool(same_elev)}")

    # --- 2) Check that the model produces the SAME probabilities given identical inputs ---
    logits_s = model({'deforestation_map': defor_single, 'elevation': elev_single})
    probs_s  = torch.sigmoid(logits_s[:, -1, 0])  # (B,H,W)

    logits_m = model({'deforestation_map': defor_multi_T, 'elevation': elev_multi})
    probs_m  = torch.sigmoid(logits_m[:, -1, 0])  # (B,H,W)

    same_probs = torch.allclose(probs_s, probs_m, atol=1e-6)
    print(f"[Check] Predicted probabilities identical (single vs multi): {bool(same_probs)}")
    print(f"[Info ] #pixels exactly == 0.5 in probs_m: {(probs_m == 0.5).sum().item()}")

    # --- 3) Compare masks/targets used in each path ---
    # Original single-year target at t+1 (with its own -1 mask)
    target_single = labels_single[:, -1, 0]  # (B,H,W) in {1,0,-1}
    valid_orig = (target_single != -1)
    print(f"[Info ] Valid pixels in original label (target != -1): {valid_orig.sum().item()}")

    # K=1 pipeline: constructed ground truth from inputs (event THIS year == 1)
    truth_k1_from_inputs = (defor_multi_all[:, T:T+1, 0] == 1).squeeze(1).to(torch.int16)  # (B,H,W)
    # Base-year mask (only evaluate where base year is forest==0)
    base_year_map = defor_multi_T[:, -1, 0]                 # (B,H,W)
    base_forest_mask = (base_year_map == 0)
    print(f"[Info ] Valid pixels in K=1 pipeline (base-year forest==0): {base_forest_mask.sum().item()}")

    # Agreement between single-year label and constructed truth (on original valid region)
    if valid_orig.any():
        agreement = (target_single[valid_orig] == truth_k1_from_inputs[valid_orig]).float().mean().item()
        print(f"[Check] Agreement(target_single vs constructed truth) on original valid region: {agreement:.6f}")
    else:
        print("[Check] No valid pixels in original label to compare.")

    # --- 4) Compute metrics in three ways (requires your original compute_metrics in scope) ---
    # A) Original single-year path: original target + original probs
    metrics_orig = compute_metrics(targets=target_single.cpu(), preds=probs_s.cpu(), dataset_name='orig')
    # B) Same target, multi path probs (should match A if inputs/probs are identical)
    metrics_k1_same_target_prob = compute_metrics(targets=target_single.cpu(), preds=probs_m.cpu(), dataset_name='k1_same_target_prob')
    # C) K=1 pipeline: constructed cumulative target + base-year mask; score probs_m against that
    targets_masked_k1 = torch.where(
        base_forest_mask,
        truth_k1_from_inputs,
        torch.full_like(truth_k1_from_inputs, -1, dtype=torch.int16)
    )
    metrics_k1_pipeline_prob = compute_metrics(targets=targets_masked_k1.cpu(), preds=probs_m.cpu(), dataset_name='k1_pipeline_prob')

    print("\n[Metrics] Original single-year (orig):")
    for k, v in metrics_orig.items():
        print(f"  {k}: {float(v):.6f}")

    print("\n[Metrics] K=1, SAME TARGET, using multi probs (k1_same_target_prob):")
    for k, v in metrics_k1_same_target_prob.items():
        print(f"  {k}: {float(v):.6f}")

    print("\n[Metrics] K=1 pipeline PROBS with constructed target + base-year mask (k1_pipeline_prob):")
    for k, v in metrics_k1_pipeline_prob.items():
        print(f"  {k}: {float(v):.6f}")

    return {
        'same_inputs': bool(same_inputs),
        'same_elevation': bool(same_elev),
        'same_probs': bool(same_probs),
        'metrics_orig': metrics_orig,
        'metrics_k1_same_target_prob': metrics_k1_same_target_prob,
        'metrics_k1_pipeline_prob': metrics_k1_pipeline_prob,
    }


In [11]:
experiments = {
    'v5kqobut' : {'name' :'52'}, #Name corresponds to random seed
    'b7pa58ky' : {'name' :'53'},
    'dndvccf8' : {'name' :'54'},
}

device = 'cuda'
for wandb_id in experiments.keys():

    print(experiments[wandb_id]['name'])

    model_path = f'../data/models_trained/{wandb_id}/best.pt'
    config_path = f'../data/models_trained/{wandb_id}/config.json'

    with open(config_path) as src:
        config = json.load(src)


    config['annotation_file'] = f"../{config['annotation_file']}"
    config['data']['deforestation_map'] = f"../{config['data']['deforestation_map']}"
    config['data']['elevation'] = f"../{config['data']['elevation']}"

    df = pd.read_csv(config['annotation_file'])
    df = df.loc[df.split == 'test']
    df['split'] = 'val'
    df.to_csv("../data/test_annotation.csv", index=False)

    config['annotation_file'] = "../data/test_annotation.csv"

        
    config['min_pos_years_val'] = 1

    config['wandb']['log_to_wandb'] = False
    loss_fn = get_loss_fn(config)
    model = get_model(config)
    loss_fn = get_loss_fn(config)

    model.load_state_dict(torch.load(model_path, weights_only=True)['state_dict'])
    _ = model.eval()

    experiments[wandb_id]['model'] = model
    experiments[wandb_id]['config'] = config
    
    
    K_steps = 1

    model_input_sequence_len = 4

    start_year = 2012
    end_year = start_year + model_input_sequence_len + K_steps + 1 # adding +1 because last item not included

    config['val_period'] = {'year_min': start_year, 'year_max': end_year, 'input_seq_len': model_input_sequence_len+K_steps}
    dataloader = get_dataloaders(['val'], config)['val']   
    
        
    _config = config.copy()
    _config['val_period'] = {'year_min': start_year, 'year_max': end_year-1, 'input_seq_len': 4}
    single_year_dtl = get_dataloaders(['val'], _config)['val']  
    

    
    # I need to do this to have only samples that have positives in 2012-2016, not 2012-2017 as is done with the multi-year dataloader
    dataloader.dataset.annotation = single_year_dtl.dataset.annotation

    
        # --- Fetch one batch from each dataloader ---
    (inp_single, labels_single) = next(iter(single_year_dtl))
    (inp_multi,  _labels_multi) = next(iter(dataloader))  # labels unused in K=1 pipeline
    assert (inp_single['deforestation_map'][:,0] == inp_multi['deforestation_map'][:,0]).all()
    _metrics = evaluate(model, loss_fn, single_year_dtl, device, _config, return_preds_and_targets=False, regression_metrics=True)



52
53
54


# Temporal tolerance

In [10]:
import torch
from typing import Dict, Tuple
from torchmetrics import MetricCollection, MeanAbsoluteError, MeanSquaredError, R2Score, PearsonCorrCoef
from torchmetrics.functional.classification import jaccard_index, f1_score, precision, recall
from sklearn.metrics import average_precision_score, roc_auc_score


def split_inputs_for_multi_year(
    input_batch: Dict[str, torch.Tensor],
    T: int,
    K: int,
    device: torch.device
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Unchanged: prepares inputs and ground truth. See your original docstring.
    """
    def_seq_all = input_batch['deforestation_map'].to(device).to(torch.float32)  # (B, T+K, 1, H, W)
    elevation   = input_batch['elevation'].to(device).to(torch.float32)         # (B, 1, H, W)
    B, X, _, H, W = def_seq_all.shape
    assert X >= T + K, f"Expected at least T+K frames, got X={X} (T={T}, K={K})."

    init_def_seq = def_seq_all[:, :T]                         # (B, T, 1, H, W)
    future_frames = def_seq_all[:, T:T+K, 0]                  # (B, K, H, W)
    future_truth_seq = (future_frames == 1).permute(1, 0, 2, 3)  # (K, B, H, W) bool
    base_year_map = init_def_seq[:, -1, 0]                    # (B, H, W)

    return init_def_seq, future_truth_seq, elevation, base_year_map


@torch.no_grad()
def predict_deforestation_within_k_years(
    model: torch.nn.Module,
    init_def_seq: torch.Tensor,
    elevation: torch.Tensor,
    K: int,
    device: torch.device,
    binarize_feedback: bool = True,
    fixed_window: bool = True
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Modified: returns BOTH the cumulative prediction and the per-year event sequence.

    Returns
    -------
    any_event_within_period : Tensor (B, H, W), float32 in {0,1}
    pred_event_seq          : Tensor (B, K, H, W), bool
        True where a pixel is newly marked as deforested at that step.
    """
    def_seq   = init_def_seq.clone().to(device).to(torch.float32)  # (B, T, 1, H, W)
    elevation = elevation.clone().to(device).to(torch.float32)     # (B, 1, H, W)
    B, T, _, H, W = def_seq.shape

    base_year_map = def_seq[:, -1, 0]                               # (B, H, W)
    ineligible_mask = (base_year_map == -1) | (base_year_map == 1)  # bool

    any_event_within_period = torch.zeros((B, H, W), device=device, dtype=torch.bool)
    pred_event_seq = torch.zeros((B, K, H, W), device=device, dtype=torch.bool)

    for step in range(K):
        # Predict probabilities for the "next year" relative to the current sequence.
        logits = model({'deforestation_map': def_seq, 'elevation': elevation})  # (B, T_current, 1, H, W)
        step_probs = torch.sigmoid(logits[:, -1, 0])                            # (B, H, W), values in [0,1]

        #logits = model({'deforestation_map': def_seq, 'elevation': elevation})  # (B, T_current, 1, H, W)
        #step_probs = torch.sigmoid(logits[:, -1, 0])                            # (B, H, W), values in [0,1]


        forest_mask   = ~ineligible_mask
        new_event_mask = (step_probs >= 0.5) & forest_mask  # bool

        any_event_within_period |= new_event_mask
        pred_event_seq[:, step] = new_event_mask

        ineligible_mask |= new_event_mask

        if binarize_feedback:
            next_frame = torch.zeros_like(base_year_map, dtype=torch.float32)
            next_frame[ineligible_mask] = -1.0
            next_frame[new_event_mask]  = 1.0
        else:
            next_frame = step_probs.clone()
            next_frame[ineligible_mask] = -1.0

        next_frame_t = next_frame.unsqueeze(1).unsqueeze(1)  # (B,1,1,H,W)
        if fixed_window:
            def_seq = torch.cat([def_seq[:, 1:], next_frame_t], dim=1)
        else:
            def_seq = torch.cat([def_seq, next_frame_t], dim=1)

    return any_event_within_period.float(), pred_event_seq  # (B,H,W), (B,K,H,W)


def compute_metrics(targets, preds, dataset_name, regression_metrics=True):
    """
    Unchanged: computes IoU/F1/Precision/Recall/AP/ROC and optional regression metrics.
    """
    preds = preds.to(dtype=torch.float16)
    targets = targets.to(dtype=torch.int16)

    mask = targets != -1
    valid_targets = targets[mask].flatten()
    valid_preds = preds[mask].flatten()

    if regression_metrics:
        binarized_preds = (preds >= 0.5).int()
        pred_pos = ((binarized_preds == 1) & mask).sum(dim=(1,2)).float()
        targ_pos = ((targets == 1) & mask).sum(dim=(1,2)).float()
        has_valid_data = mask.sum(dim=(1,2)) > 0
        pred_pos, targ_pos = pred_pos[has_valid_data], targ_pos[has_valid_data]
        pred_pos = pred_pos.unsqueeze(1)
        targ_pos = targ_pos.unsqueeze(1)
        reg_metrics_coll = MetricCollection({
            f'{dataset_name}_MAE': MeanAbsoluteError(),
            f'{dataset_name}_MSE': MeanSquaredError(),
            f'{dataset_name}_RMSE': MeanSquaredError(squared=False),
            f'{dataset_name}_R2': R2Score(),
            f'{dataset_name}_PearsonR': PearsonCorrCoef(),
        })
        reg_metrics = reg_metrics_coll(pred_pos, targ_pos)

    metrics = {
        f'{dataset_name}_iou' : jaccard_index(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_f1' : f1_score(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_precision' : precision(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_recall' : recall(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_average_precision' : average_precision_score(valid_targets.flatten(), valid_preds.flatten()) if valid_targets.numel() > 0 else float('nan'),
        f'{dataset_name}_ROC_AUC' : roc_auc_score(valid_targets.flatten(), valid_preds.flatten()) if valid_targets.numel() > 0 else float('nan'),
    }
    if regression_metrics:
        metrics.update(reg_metrics)
    return metrics


@torch.no_grad()
def evaluate_deforestation_within_k_years(
    model: torch.nn.Module,
    dataloader,
    device: torch.device,
    T: int,
    K: int,
    binarize_feedback: bool = True,
    fixed_window: bool = True,
    dataset_name: str = 'val',
    log_to_wandb: bool = False,
) -> Dict[str, torch.Tensor]:
    """
    Modified: returns metrics for ALL temporal tolerances τ ∈ {0,...,K-1} in one run.

    For each τ:
      - TP: predicted and true event exist and |t_pred - t_true| ≤ τ
      - FP: predicted event exists but not a τ-match (includes no truth OR out-of-window)
      - FN: true event exists but not a τ-match
      - TN: neither predicted nor true event
    Implemented by encoding τ into the predicted map before calling `compute_metrics`.
    """
    model.eval()
    # Accumulate per-τ preds and targets
    preds_per_tau = [ [] for _ in range(K) ]   # each element: list of (B_batch,H,W) tensors
    targs_per_tau = [ [] for _ in range(K) ]

    for input_batch, _ in dataloader:
        input_batch = {k: v.to(device).to(torch.float32) for k, v in input_batch.items()}

        # 1) Prepare inputs
        init_def_seq, future_truth_seq, elevation, base_year_map = split_inputs_for_multi_year(
            input_batch=input_batch, T=T, K=K, device=device
        )  # future_truth_seq: (K,B,H,W) bool
        truth_seq = future_truth_seq.permute(1,0,2,3).contiguous()  # (B,K,H,W) bool

        # 2) Predict per-year events + cumulative
        pred_within_period, pred_seq = predict_deforestation_within_k_years(
            model=model,
            init_def_seq=init_def_seq,
            elevation=elevation,
            K=K,
            device=device,
            binarize_feedback=binarize_feedback,
            fixed_window=fixed_window,
        )  # pred_within_period:(B,H,W), pred_seq:(B,K,H,W) bool

        # 3) Build validity mask using raw future frames
        future_frames = input_batch['deforestation_map'][:, T:T+K, 0]  # (B,K,H,W) in {1,0,-1}
        event_seq     = (future_frames == 1)     # (B,K,H,W)
        nonforest_seq = (future_frames == -1)    # (B,K,H,W)

        cum_event = torch.cumsum(event_seq.to(torch.int8), dim=1).clamp(max=1).bool()  # (B,K,H,W)
        pre_event = torch.cat([torch.zeros_like(cum_event[:, :1]), cum_event[:, :-1]], dim=1) == 0
        invalid_before_event = (nonforest_seq & pre_event).any(dim=1)  # (B,H,W)
        base_forest_mask = (base_year_map == 0)  # (B,H,W)
        valid_mask = base_forest_mask & (~invalid_before_event)  # (B,H,W)

        # 4) First-event year indices for truth and prediction
        #    Find first year where event=True; if none, assign sentinel K.
        # Truth
        truth_any = truth_seq.any(dim=1)  # (B,H,W)
        t_true = torch.full((truth_seq.shape[0], truth_seq.shape[2], truth_seq.shape[3]),
                            K, device=truth_seq.device, dtype=torch.int64)
        if truth_any.any():
            # Get indices of first True per pixel
            first_true_idx = torch.argmax(truth_seq.to(torch.int8), dim=1)
            # Mask out pixels with no event
            t_true = torch.where(truth_any, first_true_idx, t_true)

        # Prediction
        pred_any = pred_seq.any(dim=1)
        t_pred = torch.full((pred_seq.shape[0], pred_seq.shape[2], pred_seq.shape[3]),
                            K, device=pred_seq.device, dtype=torch.int64)
        if pred_any.any():
            first_pred_idx = torch.argmax(pred_seq.to(torch.int8), dim=1)
            t_pred = torch.where(pred_any, first_pred_idx, t_pred)

        # 5) Per-τ targets and preds encoding
        #    Targets are "has true event" (independent of τ).
        targets_base = truth_any.to(torch.int16)  # (B,H,W) in {0,1}
        #    For preds, mark 1 if:
        #      - no true event (so any prediction is FP and must remain 1)
        #      - OR prediction exists AND |t_pred - t_true| ≤ τ  (counts as TP)
        absdiff = torch.abs(t_pred.to(torch.int32) - t_true.to(torch.int32))  # (B,H,W), large when any=K

        for tau in range(K):
            match = pred_any & truth_any & (absdiff <= tau)       # pixels that are TP under this τ
            preds_tau = (pred_any & (~truth_any | match)).to(torch.float32)  # FP kept, TP kept, FN -> 0
            targets_tau = targets_base.clone()

            # Apply validity mask: outside → -1 on targets; preds value ignored by metrics via ignore_index
            targets_masked = torch.where(
                valid_mask,
                targets_tau,
                torch.full_like(targets_tau, -1, dtype=torch.int16)
            )

            preds_per_tau[tau].append(preds_tau.detach().cpu())
            targs_per_tau[tau].append(targets_masked.detach().cpu())


    # 6) Aggregate and compute metrics for each τ
    all_metrics = {}
    for tau in range(K):
        preds   = torch.cat(preds_per_tau[tau], dim=0)   # (N,H,W)
        targets = torch.cat(targs_per_tau[tau], dim=0)   # (N,H,W)
        metrics_tau = compute_metrics(
            targets=targets,
            preds=preds,
            dataset_name=f'{dataset_name}_by_{K}y'
        )
        
        for k,v in metrics_tau.items():
            try:
                metrics_tau[k] = round(v.item(), 4)
            except:
                pass
            
        # Store metrics under nested dictionary keyed by tau
        all_metrics[tau] = metrics_tau

    return all_metrics


In [ ]:
device = 'cuda'


In [ ]:
multiyear_pred_metrics = {}
for wandb_id in experiments.keys():
    
    
    multiyear_pred_metrics[wandb_id] = {}
    for K_steps in [1,2,3,4,5]:

        model = experiments[wandb_id]['model']
        start_year = start_year
        end_year = start_year + model_input_sequence_len + K_steps + 1 # adding +1 because last item not included

        config['val_period'] = {'year_min': start_year, 'year_max': end_year, 'input_seq_len': model_input_sequence_len+K_steps}
        dataloader = get_dataloaders(['val'], config)['val']   
        dataloader.dataset.annotation = single_year_dtl.dataset.annotation

        metrics = evaluate_deforestation_within_k_years(
            model,
            dataloader,
            device,
            model_input_sequence_len,
            K_steps)


        multiyear_pred_metrics[wandb_id][end_year -2] = metrics

        print(end_year -2)
        print(metrics)

In [13]:
multiyear_pred_metrics = {}
for wandb_id in experiments.keys():
    
    
    multiyear_pred_metrics[wandb_id] = {}
    for K_steps in [1,2,3,4,5]:

        model = experiments[wandb_id]['model']
        start_year = start_year
        end_year = start_year + model_input_sequence_len + K_steps + 1 # adding +1 because last item not included

        config['val_period'] = {'year_min': start_year, 'year_max': end_year, 'input_seq_len': model_input_sequence_len+K_steps}
        dataloader = get_dataloaders(['val'], config)['val']   
        dataloader.dataset.annotation = single_year_dtl.dataset.annotation

        metrics = evaluate_deforestation_within_k_years(
            model,
            dataloader,
            device,
            model_input_sequence_len,
            K_steps)


        multiyear_pred_metrics[wandb_id][end_year -2] = metrics

        print(end_year -2)
        print(metrics)

2016
{0: {'val_by_1y_iou': 0.302, 'val_by_1y_f1': 0.4639, 'val_by_1y_precision': 0.4567, 'val_by_1y_recall': 0.4712, 'val_by_1y_average_precision': 0.2362, 'val_by_1y_ROC_AUC': 0.7241, 'val_by_1y_MAE': 33.162, 'val_by_1y_MSE': 7501.6709, 'val_by_1y_PearsonR': 0.6295, 'val_by_1y_R2': 0.3331, 'val_by_1y_RMSE': 86.6122}}
2017
{0: {'val_by_2y_iou': 0.2279, 'val_by_2y_f1': 0.3711, 'val_by_2y_precision': 0.3912, 'val_by_2y_recall': 0.353, 'val_by_2y_average_precision': 0.183, 'val_by_2y_ROC_AUC': 0.656, 'val_by_2y_MAE': 55.4389, 'val_by_2y_MSE': 16319.9756, 'val_by_2y_PearsonR': 0.6022, 'val_by_2y_R2': 0.332, 'val_by_2y_RMSE': 127.7497}, 1: {'val_by_2y_iou': 0.3484, 'val_by_2y_f1': 0.5168, 'val_by_2y_precision': 0.4956, 'val_by_2y_recall': 0.5399, 'val_by_2y_average_precision': 0.2995, 'val_by_2y_ROC_AUC': 0.7494, 'val_by_2y_MAE': 53.4816, 'val_by_2y_MSE': 14159.5078, 'val_by_2y_PearsonR': 0.7, 'val_by_2y_R2': 0.4204, 'val_by_2y_RMSE': 118.9937}}
2018
{0: {'val_by_3y_iou': 0.1854, 'val_by_3y

2016
{0: {'val_by_1y_iou': 0.3051, 'val_by_1y_f1': 0.4675, 'val_by_1y_precision': 0.4446, 'val_by_1y_recall': 0.4929, 'val_by_1y_average_precision': 0.2393, 'val_by_1y_ROC_AUC': 0.7338, 'val_by_1y_MAE': 34.1169, 'val_by_1y_MSE': 6981.875, 'val_by_1y_PearsonR': 0.6749, 'val_by_1y_R2': 0.3794, 'val_by_1y_RMSE': 83.5576}}
2017
{0: {'val_by_2y_iou': 0.2236, 'val_by_2y_f1': 0.3655, 'val_by_2y_precision': 0.3585, 'val_by_2y_recall': 0.3727, 'val_by_2y_average_precision': 0.1771, 'val_by_2y_ROC_AUC': 0.6615, 'val_by_2y_MAE': 59.5006, 'val_by_2y_MSE': 16632.5488, 'val_by_2y_PearsonR': 0.6108, 'val_by_2y_R2': 0.3192, 'val_by_2y_RMSE': 128.9672}, 1: {'val_by_2y_iou': 0.3403, 'val_by_2y_f1': 0.5078, 'val_by_2y_precision': 0.4596, 'val_by_2y_recall': 0.5674, 'val_by_2y_average_precision': 0.2908, 'val_by_2y_ROC_AUC': 0.7588, 'val_by_2y_MAE': 59.3592, 'val_by_2y_MSE': 15166.751, 'val_by_2y_PearsonR': 0.7021, 'val_by_2y_R2': 0.3792, 'val_by_2y_RMSE': 123.1534}}
2018
{0: {'val_by_3y_iou': 0.1799, 'va

In [14]:
multiyear_pred_metrics

{'v5kqobut': {2016: {0: {'val_by_1y_iou': 0.302,
    'val_by_1y_f1': 0.4639,
    'val_by_1y_precision': 0.4567,
    'val_by_1y_recall': 0.4712,
    'val_by_1y_average_precision': 0.2362,
    'val_by_1y_ROC_AUC': 0.7241,
    'val_by_1y_MAE': 33.162,
    'val_by_1y_MSE': 7501.6709,
    'val_by_1y_PearsonR': 0.6295,
    'val_by_1y_R2': 0.3331,
    'val_by_1y_RMSE': 86.6122}},
  2017: {0: {'val_by_2y_iou': 0.2279,
    'val_by_2y_f1': 0.3711,
    'val_by_2y_precision': 0.3912,
    'val_by_2y_recall': 0.353,
    'val_by_2y_average_precision': 0.183,
    'val_by_2y_ROC_AUC': 0.656,
    'val_by_2y_MAE': 55.4389,
    'val_by_2y_MSE': 16319.9756,
    'val_by_2y_PearsonR': 0.6022,
    'val_by_2y_R2': 0.332,
    'val_by_2y_RMSE': 127.7497},
   1: {'val_by_2y_iou': 0.3484,
    'val_by_2y_f1': 0.5168,
    'val_by_2y_precision': 0.4956,
    'val_by_2y_recall': 0.5399,
    'val_by_2y_average_precision': 0.2995,
    'val_by_2y_ROC_AUC': 0.7494,
    'val_by_2y_MAE': 53.4816,
    'val_by_2y_MSE': 14159.5